In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA

from cbm import sample_mrf_prec, GaussianLangevinMechanism, MacroCausalVar, SCBM, Intervention

In [2]:
seed = 0
rs = np.random.RandomState(seed=seed)

In [3]:
def sample_from_simplex(rs, n):
    w = rs.exponential(scale=1.0, size=n)
    return w/sum(w)

In [4]:
# Define simple, linear two now SCBM
# We use the same sparsity structure within each node for simplicity. This is defined by the following binary mask:
M = np.ones((4, 4))
M[0,3] = M[1,2] = M[2,1] = M[3,0] = 0

# X1
n1 = 4
P1 = sample_mrf_prec(dim=n1, M=M, rs=rs)
mech1 = GaussianLangevinMechanism(mu=np.zeros(n1), E=np.linalg.inv(P1))
X1 = MacroCausalVar(parents=None, bottleneck_fcts=None, mechanism=mech1, n=n1)

# Bottleneck function; convex combination of inputs
w = sample_from_simplex(rs=rs, n=n1)
f_y12 = lambda x: x @ w

# X2
n2 = 4
P2 = sample_mrf_prec(dim=n2, M=M, rs=rs)
alpha = 3
mu2 = lambda x: [alpha * x_elem * np.ones(n2) for x_elem in x]
mech2 = GaussianLangevinMechanism(mu=mu2, E=np.linalg.inv(P2))
X2 = MacroCausalVar(parents=[X1], bottleneck_fcts=[f_y12], mechanism=mech2, n=n2)

In [5]:
lin_scbm = SCBM(variables=[X1, X2], seed=seed)

In [6]:
obs_sample = lin_scbm.sample(size=10000)

In [7]:
print(F'Ground truth alpha * w: {alpha * w}')

Ground truth alpha * w: [0.01144502 1.00157849 0.8437323  1.14324419]


In [8]:
# Regress X1 on X2
reg_full = LinearRegression().fit(obs_sample[0], obs_sample[1])
print(F'{reg_full.coef_}')

[[ 0.00376337  1.01869399  0.84083373  1.16559268]
 [ 0.02769491  1.00617837  0.85576916  1.15043305]
 [ 0.02589412  0.9817299   0.88275489  1.14174279]
 [-0.00163636  1.00366618  0.84089254  1.15264064]]


In [9]:
# Reduced Rank Regression
# Get principle axes of Y_hat
Y_hat = obs_sample[0] @ reg_full.coef_
pca = PCA(n_components=1)
pca.fit(Y_hat)
U = pca.components_
print(F'{reg_full.coef_ @ U.T @ U}')

[[-0.03951596  1.03777659  0.79419045  1.17876742]
 [-0.03926165  1.03109796  0.78907943  1.17118144]
 [-0.03908169  1.02637157  0.78546241  1.16581293]
 [-0.03912571  1.02752771  0.78634719  1.16712614]]
